# `tensorium` tutorial, part 4: gauge covariant derivatives

This notebook shows how `tensorium` handles covariant derivatives with internal indices. We first build a gauge connection in a fixed internal trivialization, then combine it with the Levi-Civita connection. Operator algebra is then treated in part 5.


In [1]:
from sympy import simplify, symbols
from tensorium import *

S2 = Manifold("S^2", 2)
U_N = OpenSet("U_N", S2)  # S^2 without the north pole
U_S = OpenSet("U_S", S2)  # S^2 without the south pole
x, y = symbols("x y", real=True)
u, v = symbols("u v", real=True)
X_N = Chart("X_N", U_N, (x, y))
X_S = Chart("X_S", U_S, (u, v), relations={X_N: (u/(u**2 + v**2), v/(u**2 + v**2))})
atlas = Atlas(S2, [X_N, X_S])
S2.set_atlas(atlas)
xN, yN = X_N.symbols
uS, vS = X_S.symbols
f_N_local = LocalTensorField(X_N, (0, 0), 1/(1 + xN**2 + yN**2))
f_S_local = LocalTensorField(X_S, (0, 0), (uS**2 + vS**2)/(1 + uS**2 + vS**2))
f = TensorField(S2, (0, 0), {X_N: f_N_local, X_S: f_S_local}, ())
omega_N_local = LocalTensorField(X_N, (0, 1), [xN, yN])
omega_S_local = LocalTensorField(X_S, (0, 1), [uS, vS])
omega = OneForm(S2, {X_N: omega_N_local, X_S: omega_S_local})
V_N_local = LocalTensorField(X_N, (1, 0), [yN, -xN])
V_S_local = LocalTensorField(X_S, (1, 0), [vS, -uS])
V = TensorField(S2, (1, 0), {X_N: V_N_local, X_S: V_S_local}, (1,))
lambda_N = 4/(1 + xN**2 + yN**2)**2
lambda_S = 4/(1 + uS**2 + vS**2)**2
g_N_local = LocalCovariantMetricTensor(X_N, (lambda_N, 0, lambda_N))
g_S_local = LocalCovariantMetricTensor(X_S, (lambda_S, 0, lambda_S))
g = CovariantMetricTensor(S2, {X_N: g_N_local, X_S: g_S_local})
g_inv = g.inverse()
S2_metric = MetricManifold(S2, covariant_metric=g, contravariant_metric=g_inv)
f_metric = TensorField(S2_metric, (0, 0), f.local_representations, ())
V_metric = TensorField(S2_metric, (1, 0), V.local_representations, (1,))
omega_metric = OneForm(S2_metric, omega.local_representations)
Gamma = LeviCivitaConnection(S2_metric)


## 4.1. Gauge connections

A gauge connection is represented here as a matrix-valued one-form,

$$
A = A^{[A]}{}_{[B]\mu}\,dx^\mu,
\qquad
A\in \mathcal{T}^{0}_{1}(S^2)\otimes E_2\otimes E_2^*.
$$

This is a local/trivialized description: the internal vector space is written in a fixed basis. The library does not implement non-trivial vector bundles or changes of internal frame.


In [2]:
alpha_N_local = LocalTensorField(X_N, (0, 1), [xN, yN])
alpha_S_local = LocalTensorField(X_S, (0, 1), [uS, vS])
alpha = OneForm(S2_metric, {X_N: alpha_N_local, X_S: alpha_S_local})
zero_oneform = 0*alpha

A_gauge = GaugeConnection.from_components(S2_metric, [[zero_oneform, alpha],
                                                       [-alpha, zero_oneform]])

Display(A_gauge.internal_tensor_field, name="A")

<IPython.core.display.Math object>

## 4.2. Gauge covariant derivative

Let $\xi^{[A]}$ be an internal vector. The gauge covariant derivative is

$$
(D_A\xi)^{[A]}{}_{\mu}
=
\partial_\mu\xi^{[A]}+
\sum_B A^{[A]}{}_{[B]\mu}\xi^{[B]}.
$$

In code, `CovariantDerivative(gauge_connection=A_gauge)` builds an operator with one free covariant geometric index.

<div style="font-size:0.86em; line-height:1.42; padding:0.55em 0.80em; margin:0.55em 0 0.75em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">

The operator knows, through its <code>OperatorSignature</code>, that a purely gauge covariant derivative must act on a field with internal values. If a plain <code>TensorField</code> is passed by mistake, the library raises a compatibility error before attempting to manipulate components.

</div>

In [3]:
one_metric = TensorField(S2_metric, (0, 0), {
    X_N: LocalTensorField(X_N, (0, 0), 1),
    X_S: LocalTensorField(X_S, (0, 0), 1),
}, ())

xi = TensorMultiplet([one_metric, f_metric], internal_variance=1)

D_A = CovariantDerivative(gauge_connection=A_gauge)
D_A_xi = D_A(xi)

Display(xi, name=r"\xi")
Display(D_A, name=r"D_A")
Display(D_A_xi, chart=X_N, name=r"D_A\xi")


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 4.3. Affine and gauge parts together

The same `CovariantDerivative` can combine the Levi-Civita connection and the gauge connection. If the field has both geometric and internal indices, the affine part acts on the geometric indices and the gauge part acts on the internal ones. For example,

$$
(DT)^{[A]}{}_{\nu\mu}
=
(\nabla T)^{[A]}{}_{\nu\mu}
+
\sum_B A^{[A]}{}_{[B]\mu}T^{[B]}{}_{\nu}.
$$

In [4]:
xi_omega = xi.tensor_product(omega_metric)
D_full = CovariantDerivative(affine_connection=Gamma, gauge_connection=A_gauge)
Dfull_xi_omega = D_full(xi_omega)

Display(D_full, name="D")
Display(xi_omega, chart=X_N, name=r"\xi\otimes\omega")
Display(Dfull_xi_omega, chart=X_N, name=r"D(\xi\otimes\omega)")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 4.4. Several internal indices

Covariant internal indices receive the opposite sign. For an internal matrix field $Q^{[A]}{}_{[B]}$,

$$
(DQ)^{[A]}{}_{[B]\mu}
=
\partial_\mu Q^{[A]}{}_{[B]}
+
\sum_C A^{[A]}{}_{[C]\mu}Q^{[C]}{}_{[B]}
-
\sum_C A^{[C]}{}_{[B]\mu}Q^{[A]}{}_{[C]}.
$$

This is the same rule used for gauge transformations of objects with upper and lower internal indices.


In [5]:
Q = ValuedTensorField([[one_metric, f_metric], [-f_metric, one_metric]],
                        internal_shape=(2, 2), internal_variance=(1, -1))
D_A_Q = D_A(Q)

Display(Q, chart=X_N, name="Q")
Display(D_A_Q, chart=X_N, name=r"D_A Q")


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<!-- small-note-html -->
<div style="font-size:0.92em; line-height:1.35; padding:0.45em 0.70em; margin:0.45em 0 0.70em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">
  <p style="margin:0.18em 0;">The objects constructed here, especially <code>CovariantDerivative</code>, are also tensor operators. The next notebook develops this operator point of view systematically.</p>
</div>
